# Datathon 2026 — Faz 16 HAMLE 1: TabPFN v2 OOF (tek hücre, self-contained)

**Amaç:** Yeni model ailesi (TabPFN v2, in-context tabular foundation model) ile featB'den **bağımsız** bir sinyal üret. SADECE **ham tabular** (44 sayısal + 5 kategorik) — `mentor_feedback_text` / text_meta / emb_meta / berturk_meta YOK (bağımsızlık için kasıtlı).

- Faz 1-7 ile **birebir aynı fold** (yıl×hedef-desil stratify, SEED=42, 5-fold).
- Kategorikler `categorical_features_indices` ile (label-encode, train+test birlikte).
- NaN: TabPFN v2 native destekler (impute YOK).
- `clip(0,100)`. Fold-train ~8000 satır (limit 10000, sorun yok).
- GPU gerekli (Settings → Accelerator → GPU). Model: Models → TabPFN ekle + Kabul Et (Prior-Labs/tabpfn).

**Çıktı (indirip `experiments/`e koy):**
- `/kaggle/working/oof_tabpfn_train.npy`
- `/kaggle/working/tabpfn_test.npy`

**Bana getirmen gereken:** `[TabPFN] düz=... ağırlıklı=...` satırı + yıl-bazlı kırılım (özellikle 2025-2026).

In [ ]:
# ===================== FAZ 16 / TabPFN — TEK HÜCRE =====================
import os, glob, time
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from tabpfn import TabPFNRegressor

SEED, N_SPLITS = 42, 5
ID, TARGET, TEXT, YEAR = 'student_id', 'career_success_score', 'mentor_feedback_text', 'application_year'

# ---------- Veri ----------
base = [p for p in glob.glob('/kaggle/input/*') if os.path.exists(f'{p}/train.csv')]
if not base:
    hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    assert hits, "train.csv bulunamadı — Add Input -> Datathon 2026 ekli mi?"
    base = [os.path.dirname(hits[0])]
base = base[0]
print("base:", base)
train = pd.read_csv(f'{base}/train.csv'); test = pd.read_csv(f'{base}/test_x.csv')
y = train[TARGET].values

def is_str_col(s):
    return (s.dtype == 'object' or pd.api.types.is_string_dtype(s)) and not pd.api.types.is_numeric_dtype(s)

cat_cols = [c for c in train.columns if c not in (ID, TEXT) and is_str_col(train[c])]
num_cols = [c for c in train.columns if c not in (ID, TARGET, TEXT, *cat_cols)]
print(f"train {train.shape} | test_x {test.shape} | {len(num_cols)} sayisal + {len(cat_cols)} kategorik (text YOK)")

# ---------- Fold'lar (Faz 1-7 ile birebir aynı) ----------
tbin = pd.qcut(y, 10, labels=False, duplicates='drop')
strat = train[YEAR].astype(str) + '_' + pd.Series(tbin).astype(str)
folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(train, strat))

# ---------- Test-yil-agirliklari (public proxy) ----------
tr_prop = train[YEAR].value_counts(normalize=True); te_prop = test[YEAR].value_counts(normalize=True)
w = np.nan_to_num(train[YEAR].map(lambda yr: te_prop.get(yr, 0.0) / tr_prop.get(yr, np.nan)).values)
def wmse(yt, p): return float(np.sum(w * (yt - p) ** 2) / np.sum(w))
def by_year_mse(p): return pd.DataFrame({'year': train[YEAR], 'e2': (y - p) ** 2}).groupby('year')['e2'].mean()

# ---------- Kategorikleri encode et (train+test birlikte, NaN -> -1) ----------
feat_cols = num_cols + cat_cols
X = train[feat_cols].copy(); Xt = test[feat_cols].copy()
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-1)
enc.fit(pd.concat([X[cat_cols], Xt[cat_cols]], axis=0))
X[cat_cols] = enc.transform(X[cat_cols]); Xt[cat_cols] = enc.transform(Xt[cat_cols])
cat_idx = [feat_cols.index(c) for c in cat_cols]
X = X.values.astype('float32'); Xt = Xt.values.astype('float32')
print(f"feat_cols={len(feat_cols)} | cat_idx={cat_idx}")

# ---------- TabPFN v2 OOF ----------
device = 'cuda' if os.environ.get('TABPFN_DEVICE', 'cuda') == 'cuda' else 'cpu'
oof = np.zeros(len(train)); test_pred = np.zeros(len(test))
for i, (tr, va) in enumerate(folds):
    t0 = time.time()
    m = TabPFNRegressor(device=device, categorical_features_indices=cat_idx, ignore_pretraining_limits=True, random_state=SEED)
    m.fit(X[tr], y[tr])
    oof[va] = m.predict(X[va])
    test_pred += m.predict(Xt) / len(folds)
    print(f"  fold {i}: {time.time()-t0:.0f}s")
oof = np.clip(oof, 0, 100); test_pred = np.clip(test_pred, 0, 100)

p = float(np.mean((y - oof) ** 2)); wt = wmse(y, oof)
print(f"\n================ TabPFN SONUC ================")
print(f"[TabPFN] duz={p:.4f} agirlikli={wt:.4f}")
print("\nYil-bazli (2025-2026 kritik):")
print(by_year_mse(oof).round(3).to_string())

np.save('/kaggle/working/oof_tabpfn_train.npy', oof)
np.save('/kaggle/working/tabpfn_test.npy', test_pred)
print("\n[kayit] /kaggle/working/oof_tabpfn_train.npy + tabpfn_test.npy -> indirip experiments/'e koy")